# GijsBERT training sequence — NONE silver tranche + checks

Regenerate — **do not hand-edit** the `.ipynb` JSON:

```bash
uv run python scripts/generate_notebooks.py --name gijsbert_none_silver_training
```

## What this pipeline delivers

| Stage | Output | Role |
|-------|--------|------|
| KWIC ingest | `food_snippets_long_kwic` | Deduped pool + `kwic_batch` |
| `build_kwic_inputs` | `kwic_inputs.jsonl` | Clipped windows (~±180 chars) |
| `batch_step_a` | `reizen_step_a.jsonl` | Step A only — **no Step B** on travel text |
| Review CSV | `eval/step_a_dropout_review.csv` | You mark `accept` / `reject` |
| Merge | `inception_silver_with_reizen_none.jsonl` | INCEpTION + reviewed NONE rows |
| `export_gijsbert` | `train.jsonl`, `dev.jsonl` | Train = silver, dev = **hand gold** (unchanged) |
| `train_gijsbert` | `models/gysbert-v2-…` | Fine-tuned checkpoint to compare |

**Scope:** small reviewed tranches (25–200 rows), not bulk KWIC. Dev stays 157-row hand gold (~57% NONE); train gets a few extra NONE examples.

Long LLM steps run in the **terminal** (Ollama). This notebook **checks** artifacts after each step.


## Setup — manifest paths

In [ ]:
import json
import subprocess
from collections import Counter
from pathlib import Path

import pandas as pd
from data_io import resolve

SCRATCH = Path(resolve('trifecta_gold')).parent
EVAL = Path(resolve('eval_reports'))
GIJSBERT_DIR = Path(resolve('trifecta_gijsbert'))
MODELS_DIR = Path(resolve('trifecta_gijsbert_models'))

PATHS = {
    'kwic_long': Path(resolve('food_snippets_long_kwic')),
    'kwic_inputs': Path(resolve('kwic_inputs')),
    'step_a': SCRATCH / 'reizen_step_a.jsonl',
    'dropout_review': EVAL / 'step_a_dropout_review.csv',
    'inception_silver': SCRATCH / 'inception_annotations.jsonl',
    'merged_silver': SCRATCH / 'inception_silver_with_reizen_none.jsonl',
    'train_jsonl': GIJSBERT_DIR / 'train.jsonl',
    'dev_jsonl': GIJSBERT_DIR / 'dev.jsonl',
    'label_manifest': GIJSBERT_DIR / 'label_manifest.json',
}

for name, path in PATHS.items():
    status = 'ok' if path.exists() else 'MISSING'
    print(f'{status:7} {name:16} {path}')


## Step 0 — Prerequisites

```bash
uv run python -m data_io.check
# Ollama running for Step A: ollama serve &  qwen2.5-coder:latest
```


In [ ]:
result = subprocess.run(
    ['uv', 'run', 'python', '-m', 'data_io.check'],
    capture_output=True,
    text=True,
    cwd=Path('..').resolve() if (Path.cwd() / 'scripts').exists() else Path.cwd(),
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode:
    print('WARN: manifest check returned', result.returncode)


## Step 1 — Ingest KWIC xlsx + build `kwic_inputs`

```bash
# Ingest (skip legacy CSV copies if KWIC-only refresh)
uv run python scripts/ingest_food_snippets.py --skip-csv --skip-long-csv --skip-txt

# Small reizen pool for NONE pilot (~500 rows, clipped context)
uv run python scripts/build_kwic_inputs.py --source kwic --kwic-batch reizen --limit 500
```


In [ ]:
def check_kwic_pool() -> None:
    p = PATHS['kwic_long']
    if not p.exists():
        print('SKIP: run ingest_food_snippets.py first')
        return
    df = pd.read_csv(p, usecols=['snippet', 'kwic_batch', 'matched_term'], dtype=str, nrows=50_000)
    lens = df['snippet'].str.len()
    print(f'long_kwic sample rows: {len(df):,}')
    print(f'snippet chars p50/p90: {int(lens.median()):,} / {int(lens.quantile(0.9)):,}')
    if 'kwic_batch' in df.columns:
        reizen = df['kwic_batch'].str.contains('reizen', na=False).sum()
        print(f'rows with reizen batch (sample): {reizen:,}')


def check_kwic_inputs() -> None:
    p = PATHS['kwic_inputs']
    if not p.exists():
        print('SKIP: run build_kwic_inputs.py --source kwic ...')
        return
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    lens = [len(r.get('context_text', '')) for r in rows]
    batches = Counter(b for r in rows for b in str(r.get('kwic_batch') or '').split('|') if b)
    print(f'kwic_inputs rows: {len(rows):,}')
    print(f'context_text p50 chars: {sorted(lens)[len(lens) // 2]} (expect ~300–400 after clip)')
    print('kwic_batch tags:', dict(batches.most_common(5)))


check_kwic_pool()
print()
check_kwic_inputs()


## Step 2 — Step A batch (terminal; needs Ollama)

Step A only — avoids Step B framing travel snippets as INGESTION.

```bash
uv run python scripts/batch_step_a.py \
  --input-logical kwic_inputs \
  --limit 200 --concurrency 2 --resume
```

Pilot: `--limit 25` (~30 s). Full tranche: 200 (~5–15 min with concurrency 2).


In [ ]:
def check_step_a() -> pd.DataFrame | None:
    p = PATHS['step_a']
    if not p.exists():
        print('SKIP: run batch_step_a.py (Ollama)')
        return None
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    dropped = [r for r in rows if r.get('dropped')]
    print(f'Step A rows: {len(rows)} | dropouts: {len(dropped)} ({100*len(dropped)/max(len(rows),1):.0f}%)')
    reasons = Counter(r.get('drop_reason') for r in dropped)
    print('drop_reason:', dict(reasons))
    # sample dropouts
    for r in dropped[:3]:
        prov = r.get('provenance') or {}
        a = r.get('step_a') or {}
        print('---')
        print(prov.get('target_word'), '|', r.get('drop_reason'), '|', (a.get('reasoning') or '')[:80])
        print((prov.get('context_text') or '')[:160], '…')
    return pd.DataFrame(rows)


step_a_df = check_step_a()


## Step 3 — Review dropouts (human)

```bash
uv run python scripts/export_step_a_dropout_review.py
```

Open `eval/step_a_dropout_review.csv`. Fill **`verdict`**: `accept` | `reject`.

**Review rule (NONE-scope):** accept when the target is **out of scope** for TRIFECTA frame labeling in this snippet (homograph, travel/catalog, false KWIC hit) — **not** whether Step A called it `metaphor`. Reject when real food context applies (e.g. `lever` as organ in recipe text).

**Empty `verdict` is not accept** — only `accept`, `a`, `yes`, `y`, `k`, `keep` merge.

| Column | Use |
|--------|-----|
| `recipe_context=true` | Often false NONE → scrutinize → **reject** |
| `homonym_risk` / `homonym_hint` | Pre-score from homograph heuristics |
| `step_a_metaphor` | Audit only — does **not** drive merge |
| `homonym_check` | Optional: `food_sense` / `other_sense` / `metaphor` |

Review ~50 rows/session; do **not** bulk-merge unreviewed dropouts at scale.


In [ ]:
def check_dropout_review() -> pd.DataFrame | None:
    p = PATHS['dropout_review']
    if not p.exists():
        print('SKIP: run export_step_a_dropout_review.py')
        return None
    df = pd.read_csv(p, dtype=str, keep_default_na=False)
    verdicts = df['verdict'].str.strip().str.lower()
    n_accept = verdicts.isin({'accept', 'a', 'yes', 'y', 'k', 'keep'}).sum()
    n_reject = verdicts.isin({'reject', 'r', 'no', 'n', 'drop', 'skip'}).sum()
    n_pending = len(df) - n_accept - n_reject
    print(f'review rows: {len(df)} | accept: {n_accept} | reject: {n_reject} | pending: {n_pending}')
    if n_pending:
        print('Fill verdict before merge — pending record_ids:')
        pending = df[verdicts.eq('') | (~verdicts.isin({
            'accept','a','yes','y','k','keep','reject','r','no','n','drop','skip'
        }))]
        display(pending[['record_id', 'target_word', 'recipe_context', 'drop_reason', 'context_snippet']].head(10))
    recipe_flag = df[df['recipe_context'].str.lower() == 'true']
    if len(recipe_flag):
        print(f'\nrecipe_context=true rows to scrutinize: {len(recipe_flag)}')
    return df


try:
    from IPython.display import display
except ImportError:
    display = print  # noqa: A001

review_df = check_dropout_review()


## Step 4 — Merge silver + export GijsBERT splits

```bash
uv run python scripts/merge_silver_jsonl.py \
  --base "$SCRATCH/inception_annotations.jsonl" \
  --append "$SCRATCH/reizen_step_a.jsonl" \
  --review-csv "$SCRATCH/eval/step_a_dropout_review.csv" \
  --append-limit 200 \
  --output "$SCRATCH/inception_silver_with_reizen_none.jsonl" \
  --summary

uv run python scripts/export_gijsbert.py \
  --silver-path "$SCRATCH/inception_silver_with_reizen_none.jsonl"
```

Or use `resolve()` paths from the setup cell instead of `$SCRATCH`.


In [ ]:
def label_counts_jsonl(path: Path) -> Counter:
    if not path.exists():
        return Counter()
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    return Counter(r.get('label') for r in rows)


def check_gijsbert_splits() -> None:
    if PATHS['merged_silver'].exists():
        base_n = sum(1 for _ in open(PATHS['inception_silver']) if _.strip())
        merged_n = sum(1 for _ in open(PATHS['merged_silver']) if _.strip())
        print(f'merged silver: {merged_n} rows (+{merged_n - base_n} vs inception_annotations)')
    else:
        print('merged silver: MISSING — run merge_silver_jsonl.py')

    train = label_counts_jsonl(PATHS['train_jsonl'])
    dev = label_counts_jsonl(PATHS['dev_jsonl'])
    if not train:
        print('train.jsonl: MISSING — run export_gijsbert.py')
        return
    train_n = sum(train.values())
    dev_n = sum(dev.values())
    train_none = 100 * train.get('NONE', 0) / train_n
    dev_none = 100 * dev.get('NONE', 0) / max(dev_n, 1)
    print(f'\ntrain labels ({train_n} rows):', dict(train.most_common()))
    print(f'dev labels   ({dev_n} rows):', dict(dev.most_common()))
    print(f'\nNONE %  train: {train_none:.1f}%  |  dev: {dev_none:.1f}%')
    print('Expect train NONE << dev NONE until silver grows; compare frames-only dev separately.')


check_gijsbert_splits()


## Step 5 — Fine-tune GijsBERT (terminal; `gijsbert` extra)

```bash
uv run --extra gijsbert python scripts/train_gijsbert.py \
  --output-dir "$(uv run python -c 'from data_io import resolve; print(resolve("trifecta_gijsbert_models") / "gysbert-v2-reizen-none")')" \
  --oversample-none 2 --class-weight-balance --none-weight-boost 2.0

uv run --extra gijsbert python scripts/compare_gijsbert_runs.py
```

Compare **frames-only** dev (excl. NONE) and NONE F1 — not pooled accuracy alone.


In [ ]:
def check_training_runs() -> None:
    if not MODELS_DIR.exists():
        print('No models dir:', MODELS_DIR)
        return
    runs = sorted(MODELS_DIR.iterdir())
    print('Model runs:')
    for run in runs:
        metrics = run / 'dev_metrics.json'
        flag = '✓' if metrics.exists() else ' '
        print(f'  [{flag}] {run.name}')
        if metrics.exists():
            m = json.loads(metrics.read_text())
            acc = m.get('accuracy') or m.get('eval_accuracy')
            print(f'       pooled acc: {acc}')
            if 'per_class' in m:
                none = m['per_class'].get('NONE', {})
                print(f'       NONE F1: {none.get("f1")}')


check_training_runs()


## Decision checklist (before scaling NONE silver)

1. **Dropout quality** — mostly homograph noise (`mede`, canal `water`), not recipe CURE (`lever`)?
2. **Train NONE %** — moved toward dev, or still ~12% vs ~57%?
3. **Dev NONE F1** — improved vs `gysbert-v2-balanced-none` / `none-oversample-10`?
4. **Frames-only dev** — did extra NONE hurt frame discrimination?

Pilot (Jul 2026): `gysbert-v2-reizen-none` — 61.8% pooled dev, NONE F1 0.82; frames-only ~35%. NONE silver loop **validated**; defer further tranches unless targeting NONE at scale.

Docs: [SNIPPETS.md](../docs/SNIPPETS.md) · [ANNOTATION_STRATEGY.md](../docs/ANNOTATION_STRATEGY.md)


## State of affairs (10 Jul 2026)

**Primary goal:** qualia for analysis (Step C), not more NONE silver or GijsBERT-as-replacement for qwen.

### What works today

| Capability | Status |
|------------|--------|
| Full pipeline A→B→C (`trifecta-annotate`, `trifecta-batch`) | **Production-ready** (LLM) |
| Step B eval on frozen 157-row gold | **75%** qwen baseline; per-regime reporting |
| Step A NONE silver loop (`reizen` pilot) | **Validated** — 200 Step A → 59 accepted |
| GijsBERT macro-frame classifier | **Trained** — 61.8% pooled dev, NONE F1 0.82 |
| INCEpTION silver with Step C | ~2,752 LLM `step_c` rows (exploration only) |

### Gaps for qualia-for-analysis

| Gap | Current |
|-----|---------|
| Hand gold with Step C | **13 / 157** rows |
| Step C in `trifecta-eval` | **Not implemented** |
| GijsBERT qualia | **Not planned** — Step C stays LLM |

### Usable for analysis?

- **Exploratory qualia:** yes — `trifecta-batch` on a KWIC slice; treat `step_c` as LLM hypotheses.
- **Analytic claims (counts, trends):** not yet — need quota hand gold + Step C eval.

See [PLAN.md](../PLAN.md) · milestone **m10**.


## Next — qualia for analysis (m10)

1. **Quota gold Step C** — one frame first (PRESERVING or COOKING_CREATION): ~30–50 rows, all qualia fields.
2. **Batch + eval** — `trifecta-batch` on gold slice; add Step C field-match to `trifecta-eval`.
3. **Prompt / few-shot pass** — tune lowest-F1 qualia roles; re-run on gold only.
4. **Analysis batch** — full A→B→C on bounded KWIC export → parquet for notebooks.

```bash
# Exploratory qualia now (LLM hypotheses — spot-check before aggregating)
uv run trifecta-batch --input-logical kwic_inputs --output-logical trifecta_annotations --resume --concurrency 2

# Gold labelling for Step C fields
uv run python scripts/export_gold_candidates.py --limit 50
```


In [ ]:
def check_qualia_coverage() -> None:
    from data_io import load_parquet, load_jsonl

    gold = load_parquet('trifecta_gold')
    n_gold = len(gold)
    with_b = with_c = 0
    frames: Counter[str] = Counter()
    for raw in gold['annotation_json']:
        ann = json.loads(raw) if isinstance(raw, str) else raw
        if ann.get('step_b'):
            with_b += 1
            f = (ann.get('step_b') or {}).get('selected_frame', '?')
            frames[f] += 1
        if ann.get('step_c'):
            with_c += 1
    print(f'hand gold: {n_gold} rows | step_b: {with_b} | step_c: {with_c}')
    print('gold frames:', dict(frames))

    silver_path = PATHS['inception_silver']
    if silver_path.exists():
        rows = load_jsonl(silver_path)
        sc = sum(1 for r in rows if r.get('step_c'))
        framed = sum(1 for r in rows if r.get('step_b') and not r.get('dropped'))
        print(f'\nINCEpTION silver: {len(rows)} rows | framed: {framed} | step_c: {sc} (LLM, unvalidated)')

    ann_path = Path(resolve('trifecta_annotations'))
    if ann_path.exists():
        rows = load_jsonl(ann_path)
        sc = sum(1 for r in rows if r.get('step_c'))
        print(f'batch output: {len(rows)} rows | step_c: {sc}')
    else:
        print('\nbatch output: MISSING — run trifecta-batch for exploratory qualia')


check_qualia_coverage()
